<a href="https://colab.research.google.com/github/si66326h-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si66326h-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
### Plain-Words Contract
* **Unit of Analysis (Grain):** One row represents performance metrics for one content item (`content_id`) for one client (`client_id`) on a single calendar date (`report_date`).
* **Tables Used:** `fact_content_daily_performance` from `hf://datasets/FlyRank/internship-warehouse`.
* **Time Window:** Mid-panel month of **March 2026** (`month=2026-03`, dates `2026-03-01` to `2026-03-31`). *(Note: `_sample` is avoided as it contains the sealed test month of June 2026).*
* **Target / Label:** Predicting `gsc_clicks` (daily traffic performance).
* **Exclusions:** Filter out rows where `ga4_data_available IS FALSE` because GA4 metrics are zero-filled prior to client integration, introducing false zero-engagement signals.

In [13]:
import duckdb
from google.colab import userdata

# 1. Retrieve Hugging Face token safely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect to DuckDB and load the httpfs extension
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# 3. Create S3 Secret for Hugging Face authentication
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE S3,
        KEY_ID 'bearer',
        SECRET '{hf_token}',
        REGION 'us-east-1'
    );
""")

print("DuckDB connected and authenticated with Hugging Face!")

DuckDB connected and authenticated with Hugging Face!


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Feature Bucket:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`.
* **Label Bucket:** `target_today_clicks` (today's actual GSC clicks).
* **Context Bucket:** `client_id`, `content_id`, `report_date`.
* **Excluded Bucket:** Rows where `ga4_data_available IS FALSE` (explained above).

In [14]:
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download, snapshot_download

# 1. Retrieve Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Download the March 2026 dataset files locally using the Hugging Face Hub SDK
local_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns="fact_content_daily_performance/month=2026-03/*",
    token=hf_token
)

# 3. Initialize DuckDB
con = duckdb.connect()

print("Files successfully downloaded and authenticated!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Files successfully downloaded and authenticated!


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# Query 1: Grain Uniqueness Check (Expected 0 rows)
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM '{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5;
""").df()
print("1. Grain Violations (Expected 0):", len(grain_check))

# Query 2: Row Count & Date Span
span_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM '{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet';
""").df()
print("\n2. Row Count & Span:")
print(span_check)

# Query 3: Availability Check (IS TRUE)
avail_check = con.execute(f"""
    SELECT
        COUNT(*) AS valid_ga4_rows,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM '{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet'), 2) AS pct_survived
    FROM '{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE;
""").df()
print("\n3. GA4 Data Availability:")
print(avail_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain Violations (Expected 0): 0

2. Row Count & Span:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

3. GA4 Data Availability:
   valid_ga4_rows  pct_survived
0          413966          4.21


In [17]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# Fetch 5 non-leaky features (1-day lags) using correct column names
df = con.execute(f"""
    SELECT
        report_date, client_hash_id, content_hash_id,
        LAG(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS lag_1d_clicks,
        LAG(gsc_impressions, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS lag_1d_impressions,
        LAG(gsc_avg_position, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS lag_1d_position,
        LAG(ga4_sessions, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS lag_1d_sessions,
        LAG(ga4_engaged_sessions, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS lag_1d_engaged,
        gsc_clicks AS target_today_clicks
    FROM '{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE
""").df().dropna()

# Feature 1: lag_1d_clicks — Knowable at T because it uses GSC clicks up to T-1.
# Feature 2: lag_1d_impressions — Knowable at T because it uses GSC impressions up to T-1.
# Feature 3: lag_1d_position — Knowable at T because it uses average rank up to T-1.
# Feature 4: lag_1d_sessions — Knowable at T because it uses GA4 sessions up to T-1.
# Feature 5: lag_1d_engaged — Knowable at T because it uses engaged sessions up to T-1.

# TRAP: Inject target-derived column
df['leaky_clicks'] = df['target_today_clicks'] * 1.0

# Evaluate WITH leak
X_leak = df[['lag_1d_clicks', 'lag_1d_impressions', 'lag_1d_position', 'lag_1d_sessions', 'lag_1d_engaged', 'leaky_clicks']]
y = df['target_today_clicks']
score_leak = r2_score(y, Ridge().fit(X_leak, y).predict(X_leak))
print(f"R² WITH Leak: {score_leak:.4f}")

# Evaluate WITHOUT leak
X_honest = df[['lag_1d_clicks', 'lag_1d_impressions', 'lag_1d_position', 'lag_1d_sessions', 'lag_1d_engaged']]
score_honest = r2_score(y, Ridge().fit(X_honest, y).predict(X_honest))
print(f"R² WITHOUT Leak (Honest Baseline): {score_honest:.4f}")

R² WITH Leak: 1.0000
R² WITHOUT Leak (Honest Baseline): 0.6814


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
### Named Data Limitation
**Unbalanced History & GSC Query Anonymization:** Client onboarding dates vary across `dim_clients`, leading to varying historical depths per client. Additionally, Google Search Console anonymizes low-volume search queries, meaning aggregate page clicks cannot always be fully broken down by individual keyword queries.

In [19]:
import duckdb
from google.colab import userdata
from huggingface_hub import snapshot_download

# 1. Retrieve Hugging Face Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Download March 2026 Parquet files directly to Colab disk
local_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns="fact_content_daily_performance/month=2026-03/*",
    token=hf_token
)

# 3. Initialize DuckDB Connection
con = duckdb.connect()

print("Files successfully downloaded and authenticated with Hugging Face!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Files successfully downloaded and authenticated with Hugging Face!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.